# 2.5 Neural Network Fitting Benchmarks

This notebook runs the full benchmark sweep of neural-network SHO fitting:
optimizers (Adam, Trust-Region CG) x noise levels (0-8) x batch sizes x seeds.

Notes:
- The full grid (360 training runs) is designed for a GPU (e.g., Google Colab).
- Noise levels 1-8 require the noisy datasets *and their LSQF SHO fits* generated by
  running notebook `0_5_Noisy_Data_and_Fitting.ipynb` at full scale (`QUICK_RUN = False`).
- With `QUICK_RUN = True` a single small training run on the raw data (noise 0) is executed
  to verify the pipeline end-to-end on a CPU.

In [ ]:
# --- Environment setup (works on Colab and locally) ---
REPO_URL = "https://github.com/m3-learning/m3_learning.git"
BRANCH = "shofit"
QUICK_RUN = False  # True = small subset / few epochs, for fast verification

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import importlib.util, subprocess
    if importlib.util.find_spec("m3_learning") is None:
        subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1",
                        REPO_URL, "/content/m3_learning_repo"], check=True)
        subprocess.run(["pip", "install", "-q",
                        "/content/m3_learning_repo/m3_learning"], check=True)
        subprocess.run(["pip", "install", "-q", "--upgrade", "numpy_groupies"], check=True)
    subprocess.run(["pip", "install", "-q", "dataerai-cli-beta==0.1.54", "dataerai-sdk[notebook]==0.2.0b52", "keyring"], check=True)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Colab: {IN_COLAB}, device: {device}")

## Dataerai notebook and neural-network provenance

Authenticate once with `dataerai auth login --device --client-id dataerai-mobile --server https://beta.dataerai.com`. The next cell starts `%dataerai --trace` and the M3 artifact publisher. Together they record cell execution; reuse source datasets; publish figures, changed HDF5/CSV data, checkpoints, loss histories, and manifests; and link those products to the notebook run and source data. Set `DATAERAI_DESTINATION_COLLECTION_PATH` before launching Jupyter to override the default destination.


In [ ]:
import os as _dataerai_os
import subprocess as _dataerai_subprocess
import sys as _dataerai_sys

_dataerai_subprocess.run(
    [
        _dataerai_sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--pre",
        "dataerai-cli-beta==0.1.54",
        "dataerai-sdk[notebook]==0.2.0b52",
    ],
    check=True,
)

try:
    from m3_learning.artifacts import start_dataerai_artifact_publishing
except ImportError:
    _dataerai_subprocess.run(
        [
            _dataerai_sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "--no-deps",
            "git+https://github.com/m3-learning/"
            "m3_learning.git@codex/dataerai-notebook-training-provenance"
            "#subdirectory=m3_learning",
        ],
        check=True,
    )
    from m3_learning.artifacts import start_dataerai_artifact_publishing

DATAERAI_NOTEBOOK = '2_5_nn_fitting_all.ipynb'
DATAERAI_NOTEBOOK_TITLE = '2.5 Neural Network Fitting Benchmarks'
DATAERAI_DESTINATION_COLLECTION_PATH = _dataerai_os.environ.get(
    "DATAERAI_DESTINATION_COLLECTION_PATH",
    'M3 Learning / Notebook Provenance / papers / 2023 Rapid Fitting',
)

%load_ext dataerai.magics
%dataerai --request-timeout 120 --trace --notebook "$DATAERAI_NOTEBOOK" --title "$DATAERAI_NOTEBOOK_TITLE" "$DATAERAI_DESTINATION_COLLECTION_PATH"

_dataerai_os.environ["DATAERAI_NOTEBOOK_TRACE_RUN_ID"] = dataerai_session.trace_run_id
_dataerai_os.environ["DATAERAI_NOTEBOOK_COLLECTION_PATH"] = dataerai_session.collection_path
dataerai_artifacts = start_dataerai_artifact_publishing(
    dataerai_session,
    DATAERAI_NOTEBOOK,
    shell=get_ipython(),
)


## Dataerai login and provenance logging

Run these commands once before enabling lineage logging. On macOS, `dataerai auth login` is all you need: credentials are read from the system keychain via the `keyring` package. On Colab or headless machines, use device login.

```bash
python -m pip install --pre dataerai-cli-beta==0.1.54 'dataerai-sdk[notebook]==0.2.0b52' keyring
dataerai auth login --device --client-id dataerai-mobile --server https://beta.dataerai.com
dataerai auth status
dataerai upload --project <project-uuid> --collection <collection-uuid> --metadata dataerai_source_asset.json --file Data/data_raw.h5
export DATAERAI_DATASET_ASSET_ID=<asset-id>
export DATAERAI_LOG_PROVENANCE=1
```

Leave `DATAERAI_LOG_PROVENANCE` unset to run without network logging. If you already know the provenance surrogate, set `DATAERAI_DATASET_RECORD_SK` instead of `DATAERAI_DATASET_ASSET_ID`.


In [ ]:
import os

DATAERAI_NOTEBOOK = '2_5_nn_fitting_all.ipynb'
LOG_DATAERAI_PROVENANCE = os.getenv("DATAERAI_LOG_PROVENANCE", "").lower() in {"1", "true", "yes", "on"}
DATAERAI_DATASET_ASSET_ID = os.getenv("DATAERAI_DATASET_ASSET_ID") or None
DATAERAI_DATASET_RECORD_SK = os.getenv("DATAERAI_DATASET_RECORD_SK") or None

if DATAERAI_DATASET_ASSET_ID or DATAERAI_DATASET_RECORD_SK:
    LOG_DATAERAI_PROVENANCE = True

print("Dataerai provenance logging:", "enabled" if LOG_DATAERAI_PROVENANCE else "disabled")


In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
import os
from datetime import datetime

import numpy as np

from m3_learning.be.nn import SHO_fit_func_nn, SHO_Model, batch_training
from m3_learning.util.system_info import SystemInfo
from m3_learning.util.file_IO import download_and_unzip
from m3_learning.optimizers.TrustRegion import TRCG
from m3_learning.nn.random import random_seed
from m3_learning.viz.style import set_style
from m3_learning.viz.printing import printer
from m3_learning.be.viz import Viz
from m3_learning.be.dataset import BE_Dataset

# Note: on multi-GPU machines you can pin a specific GPU, e.g.:
# os.environ['CUDA_VISIBLE_DEVICES'] = '1'

print(f'using torch version {torch.__version__}')

printing = printer(basepath='./Figures/')

set_style("printing")
random_seed(seed=42)

In [ ]:
# Download the data file from Zenodo (skipped if the file already exists,
# e.g. if you already ran notebooks 0_5, 1 or 2)
url = 'https://zenodo.org/record/7774788/files/PZT_2080_raw_data.h5?download=1'

# Specify the filename and the path to save the file
filename = 'data_raw.h5'
save_path = './Data'

# download the file
download_and_unzip(filename, url, save_path)

data_path = save_path + "/" + filename

In [ ]:
# Trust-Region CG optimizer configuration (device-agnostic)
optimizer_TR = {"name": "TRCG", "optimizer": TRCG, "radius": 5,
                "device": device, "ADAM_epochs": 2}

if QUICK_RUN:
    # Minimal grid for fast CPU verification: a single Adam run on the raw data.
    # Noise levels 1-8 are excluded because they need the noisy-data LSQF SHO fits
    # produced by running notebook 0_5 at full scale.
    optimizers = ['Adam']
    noise_list = [0]
    batch_size = [1000]
    epochs = [1]
    seed = [41]
    early_stopping_time = 60
    max_train_size = 10000  # subsample the training data
else:
    # Full benchmark grid exactly as run in the paper (360 training runs)
    optimizers = ['Adam', optimizer_TR]
    noise_list = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    batch_size = [500, 1000, 5000, 10000]
    epochs = [5]
    seed = [41, 43, 44, 45, 46]
    early_stopping_time = 60 * 3
    max_train_size = None

basepath_postfix = 'nn_benchmarks_noise'

In [ ]:
# instantiate the dataset object
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# computes the SHO LSQF fit if it has not been performed previously.
# This is cached/idempotent: it returns instantly if you already ran notebook 1 or 2.
dataset.SHO_Fitter(force=False, h5_sho_targ_grp="Raw_Data_SHO_Fit")

# reinstantiate the dataset object after fitting
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# print the contents of the file
dataset.print_be_tree()

# Get the current date and time used to name the benchmark output folder
# Format the date and time in a 'pretty' format (e.g., YYYY-MM-DD_HH-MM-SS)
formatted_datetime = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

In [ ]:
basepath = f'{formatted_datetime}_{basepath_postfix}'

# records the system information (CPU/GPU) used for the benchmark
system_info = SystemInfo()
cpu_info, gpu_info = system_info.get_system_info()
system_info.save_to_file("Trained Models/" + basepath,
                         "system_info.txt", cpu_info, gpu_info)

In [ ]:
# Generate all combinations
combinations = list(itertools.product(
    optimizers, noise_list, batch_size, epochs, seed))

for i, training in enumerate(combinations):

    optimizer_ = training[0]
    noise_ = training[1]
    seed_ = training[4]

    print(i, optimizer_, noise_, seed_)

In [ ]:
batch_training(dataset, optimizers, noise_list, batch_size, epochs,
               seed,
               write_CSV="Batch_Trainging_SpeedTest.csv",
               basepath=basepath,
               early_stopping_time=early_stopping_time,
               max_train_size=max_train_size,
               log_dataerai_provenance=LOG_DATAERAI_PROVENANCE,
               dataerai_dataset_asset_id=DATAERAI_DATASET_ASSET_ID,
               dataerai_dataset_record_sk=DATAERAI_DATASET_RECORD_SK,
               dataerai_extra_params={"notebook": DATAERAI_NOTEBOOK})

## Finish the Dataerai provenance record

This cell first uploads captured figures, changed HDF5/CSV files, and model artifacts. It then publishes the single notebook execution log and its `records_telemetry` relationships.


In [ ]:
dataerai_artifact_result = dataerai_artifacts.finish()
%dataerai --finish
